<a href="https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/AI%E6%8F%90%E5%95%8F%E8%88%87%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92%E5%AF%A6%E6%88%B0/01_%E5%BB%A3%E5%91%8A%E6%95%88%E7%9B%8A%E9%A0%90%E6%B8%AC_%E8%BF%B4%E6%AD%B8%E7%AF%87/%E5%BB%A3%E5%91%8A%E6%95%88%E7%9B%8A%E9%A0%90%E6%B8%AC_%E8%BF%B4%E6%AD%B8%E7%AF%87.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
# 若在 Google Colab 上執行，請先安裝中文字型支援庫與下載工具
%pip install wget mplfonts


# 📈 範例 01：廣告效益預測 (迴歸篇) — AI 伴學實戰筆記本

歡迎來到本實戰筆記本！本筆記本採用**「一段程式碼 + 一段 AI 互動學習指引」**的設計架構：
- 每一段程式碼都經過嚴格測試，保證可以直接執行。
- 執行完程式碼後，**下方的文字儲存格**會詳細解釋程式在做什麼，並提供專屬的 **Prompt（提示詞）**，讓你可以複製貼給 AI，請 AI 替你解說細節、深入探討觀念、或幫你修改程式碼！

---

## 📊 一、資料集介紹 (`advSale.csv`)

👉 **資料集直接下載**：[advSale.csv (點此下載或於新分頁開啟)](https://raw.githubusercontent.com/roberthsu2003/machine_learning/main/AI%E6%8F%90%E5%95%8F%E8%88%87%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92%E5%AF%A6%E6%88%B0/01_%E5%BB%A3%E5%91%8A%E6%95%88%E7%9B%8A%E9%A0%90%E6%B8%AC_%E8%BF%B4%E6%AD%B8%E7%AF%87/advSale.csv)

本數據集記錄了某企業在 4,500 個行銷週期中，各廣告媒體的預算投入與最終產品銷售額：

| 欄位名稱 | 資料型態 | 單位 / 數值範例 | 業務意義 |
| :--- | :--- | :--- | :--- |
| **電視廣告** | 數值型 (`float64`) | 93.0 (千元) | 電視廣告費用 |
| **廣播廣告** | 數值型 (`float64`) | 25.6 (千元) | 廣播電台廣告費用 |
| **社群媒體廣告** | 數值型 (`float64`) | 5.1 (千元) | FB/IG 等網路社群廣告費用 |
| **網紅廣告** | 類別型 (`object`) | 「大於百萬」、「小於一萬」 | 合作網紅粉絲規模級距 |
| **銷售額 (目標 y)** | 數值型 (`float64`) | 334.5 (萬元) | **我們想預測的目標連續變數** |

---

## 🧭 二、機器學習全流程地圖

我們將帶領你走過專業資料科學家的標準 6 大步驟：
1. **環境配置與字型**：解決繪圖中文方塊亂碼問題。
2. **資料載入與探索 (EDA)**：檢查形狀、缺失值與資料前幾筆長相。
3. **相關性分析與視覺化**：繪製熱力圖，找出誰是帶動銷售的關鍵管道。
4. **特徵工程與 Pipeline 建置**：切分 80% 訓練與 20% 測試集，利用管線封裝防止資料洩漏。
5. **模型選型與交叉驗證比較**：比較「多元線性迴歸 (Baseline)」與「隨機森林迴歸」。
6. **測試集大考與業務預測**：解讀 R²、RMSE 與 MAE，並進行新預算方案試算。

---  
### 【步驟 0】：環境準備與繁體中文字型設定
在開始之前，我們先載入標準資料科學套件，並設定好繁體中文字型，確保後續繪圖不會出現豆腐塊亂碼 (`□□`)。

In [ ]:
# [步驟 0 程式碼] 載入套件與設定中文字型
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 自動設定繁體中文字型（支援 mplfonts 與各作業系統系統字型）
try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Microsoft JhengHei", "sans-serif"]
    plt.rcParams["axes.unicode_minus"] = False

print("✅ 套件與中文字型載入成功！")

#### 💬【步驟 0 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  載入 Pandas（資料處理）、NumPy（矩陣運算）、Matplotlib/Seaborn（資料視覺化），並利用 `mplfonts` 或作業系統原生字型設定中文字型支援，徹底解決資料科學繪圖時中文變亂碼方塊 (`□□`) 的經典痛點。

---
💡 **學生應該掌握的核心觀念與 AI 提問清單（依你的學習需求自由複製發問）：**

1. 📌 **中文字型與編碼原理**
   > 「請解釋為什麼 Matplotlib 在繪製中文圖表時預設會出現亂碼或方塊 (`□□`)？其底層字型庫的運作機制是什麼？」

2. 📌 **負號顯示異常排查**
   > 「程式碼中的 `plt.rcParams[axes.unicode_minus] = False` 具體是在解決什麼問題？如果不加這行，圖表座標軸出現負數時會發生什麼錯誤？」

3. 📌 **跨作業系統環境適配與改寫**
   > 「如果我換到 Windows、macOS 或 Linux 環境，除了使用 `mplfonts` 套件外，還有哪些系統自帶的字型（例如微軟正黑體、蘋方）可以設定給 Matplotlib？請提供可相容的寫法。」


---  
### 【步驟 1】：載入資料集與檢視基本結構
執行下方儲存格，讀入 `advSale.csv`，檢視總筆數、欄位資料型態，以及前 5 筆樣本數據。

In [ ]:
# [步驟 1 程式碼] 讀取資料與結構探索
import os

# 若本地不存在 advSale.csv（例如在 Google Colab 環境中執行），自動從 GitHub 下載
if not os.path.exists("advSale.csv"):
    import wget
    print("正在下載 advSale.csv 資料集...")
    wget.download("https://raw.githubusercontent.com/roberthsu2003/machine_learning/main/AI%E6%8F%90%E5%95%8F%E8%88%87%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92%E5%AF%A6%E6%88%B0/01_%E5%BB%A3%E5%91%8A%E6%95%88%E7%9B%8A%E9%A0%90%E6%B8%AC_%E8%BF%B4%E6%AD%B8%E7%AF%87/advSale.csv")
    print("\n✅ 下載完成！")

df = pd.read_csv("advSale.csv")

print("=== 資料結構與缺失值摘要 ===")
df.info()

print("\n=== 前 5 筆資料樣本 ===")
display(df.head())

print("\n=== 網紅廣告各級距分佈統計 ===")
print(df["網紅廣告"].value_counts())


#### 💬【步驟 1 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  `pd.read_csv` 將 CSV 讀取為表格物件 DataFrame。`df.info()` 顯示共有 4,500 筆資料且無任何缺失值（Non-Null Count 全為 4,500）。`value_counts()` 統計了「網紅廣告」中四種文字級距的分佈筆數。

---
💡 **學生應該掌握的核心觀念與 AI 提問清單（依你的學習需求自由複製發問）：**

1. 📌 **特徵與目標變數定義 (Features vs. Target)**
   > 「在這份 4,500 筆的廣告數據中，什麼是『特徵變數 (Feature / X)』？什麼是『目標變數 (Target / y)』？這屬於機器學習中的『迴歸 (Regression)』還是『分類 (Classification)』問題？為什麼？」

2. 📌 **缺失值探索與實務處理策略**
   > 「`df.info()` 顯示資料非常乾淨沒有缺失值。但在真實業界專案中，若欄位出現缺失值 (NaN/None)，常見的補值策略有哪些（如刪除、平均值、中位數、眾數補齊）？各會有什麼優缺點？」

3. 📌 **數值型 vs 類別型變數的本質差異**
   > 「『電視廣告』是連續浮點數，而『網紅廣告』是文字字串（大於百萬、小於一萬等）。為什麼機器學習演算法無法直接處理文字？如果不做轉換直接丟給模型會發生什麼錯誤？」

4. 🛠️ **進階探索改寫 Prompt**
   > 「請幫我寫一段 Pandas 程式碼，使用 `.describe()` 算出數值欄位的平均值、標準差與四分位距，並用 Seaborn 繪製『銷售額』的分佈直方圖 (Histogram) 與核密度圖 (KDE)。」


---  
### 【步驟 2】：特徵相關性分析與視覺化熱力圖 (EDA)
執行下方儲存格，計算各數值型廣告費用與銷售額之間的皮爾森相關係數 (Correlation)，並繪製相關性熱力圖。

In [ ]:
# [步驟 2 程式碼] 相關係數計算與熱力圖繪製
num_cols = ["電視廣告", "廣播廣告", "社群媒體廣告", "銷售額"]
corr = df[num_cols].corr()

print("=== 各廣告與銷售額之相關係數排序 ===")
print(corr["銷售額"].sort_values(ascending=False))

# 繪製熱力圖
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="Blues", fmt=".2f", cbar=True)
plt.title("廣告媒體預算與銷售額之相關性熱力圖", fontsize=13)
plt.tight_layout()
plt.show()

#### 💬【步驟 2 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  計算欄位兩兩之間的皮爾森相關係數（介於 -1 到 +1 之間）。熱力圖直觀顯示「電視廣告」與「銷售額」的係數高達 0.99，代表兩者呈現極強的線性正相關！而「廣播廣告」為 0.25，「社群廣告」僅 0.08。

---
💡 **學生應該掌握的核心觀念與 AI 提問清單（依你的學習需求自由複製發問）：**

1. 📌 **皮爾森相關係數解讀**
   > 「皮爾森相關係數 (Pearson Correlation) 的數學意涵是什麼？+1、0、-1 分別代表什麼意思？熱力圖中『電視廣告』與『銷售額』高達 0.99 代表什麼商業訊息？」

2. 📌 **相關性不等於因果性 (Correlation vs. Causation)**
   > 「統計學常說『相關性不等於因果性』。請用這份廣告數據為例，向我解釋為什麼我們不能單憑相關係數 0.99，就武斷認定是電視廣告『直接造成』了銷售額增長？」

3. 📌 **低相關性變數的決策迷思**
   > 「熱力圖中『社群媒體廣告』與銷售額的相關係數只有 0.08，這代表社群廣告完全無效、應該立即砍掉預算嗎？是否有可能是非線性關係，或是受到網紅級距的協同影響？」

4. 🛠️ **進階探索改寫 Prompt**
   > 「請幫我修改這段程式碼，改為使用 Seaborn 的 `regplot` 繪製『電視廣告 vs 銷售額』與『廣播廣告 vs 銷售額』的散佈圖，並在上面繪製紅色的最佳擬合線性趨勢線。」


---  
### 【步驟 3】：特徵工程與 Pipeline 建置 (嚴防資料洩漏)
執行下方儲存格，切分特徵 $X$ 與目標 $y$，切分訓練集 (80%) 與測試集 (20%)，並利用 `ColumnTransformer` 將數值標準化與類別獨熱編碼封裝。

In [ ]:
# [步驟 3 程式碼] 資料切分與特徵前處理管線
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. 切分特徵 X 與目標 y
X = df.drop(columns=["銷售額"])
y = df["銷售額"]

# 2. 定義前處理器：數值欄位做 StandardScaler，文字欄位做 OneHotEncoder
numeric_features = ["電視廣告", "廣播廣告", "社群媒體廣告"]
categorical_features = ["網紅廣告"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first", sparse_output=False), categorical_features)
    ]
)

# 3. 切分訓練集 (80%) 與測試集 (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"✅ 資料切分完成！訓練集樣本數: {X_train.shape[0]}, 測試集樣本數: {X_test.shape[0]}")

#### 💬【步驟 3 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  透過 `train_test_split` 將資料切分為 80% 訓練集與 20% 測試集。更重要的是，使用 `ColumnTransformer` 定義特徵轉換器（數值做 `StandardScaler`、文字做 `OneHotEncoder`），將特徵工程規格封裝起來。

---
💡 **學生應該掌握的核心觀念與 AI 提問清單（依你的學習需求自由複製發問）：**

1. 📌 **切分訓練集與測試集的根本目的**
   > 「為什麼機器學習必須切分出訓練集與測試集？如果把 100% 的資料全部拿去訓練模型，會引發什麼嚴重的模型評估問題？」

2. 📌 **資料洩漏 (Data Leakage) 致命陷阱**
   > 「為什麼 `StandardScaler` 不能在切分資料前對整份 DataFrame 執行？為什麼必須『在訓練集 fit，再 transform 到測試集』？請用白話文解釋什麼是資料洩漏 (Data Leakage)。」

3. 📌 **獨熱編碼與虛擬變數陷阱 (Dummy Variable Trap)**
   > 「什麼是 One-Hot Encoding（獨熱編碼）？為什麼文字類別不能直接填 1, 2, 3, 4？參數中的 `drop=first` 是在防範統計學上的什麼陷阱（多重共線性）？」

4. 📌 **Pipeline 與 ColumnTransformer 的工程優勢**
   > 「在企業級機器學習專案中，為什麼資深工程師強烈推薦使用 `ColumnTransformer` 與 `Pipeline`，而不是手動對 DataFrame 欄位逐一做轉換？它對未來模型部署有何好處？」

5. 🛠️ **進階探索改寫 Prompt**
   > 「我想把測試集改為 25%，並且在前處理管線中加入『數值缺失值中位數補值器 (SimpleImputer)』以防未來新資料出現空值，這段程式該如何撰寫？」


---  
### 【步驟 4】：模型選型與 5 折交叉驗證比較 (Baseline vs. 進階模型)
執行下方儲存格，建立「多元線性迴歸 (Baseline)」與「隨機森林迴歸 (Random Forest)」，並透過 5 折交叉驗證評估穩定度。

In [ ]:
# [步驟 4 程式碼] Pipeline 整合與交叉驗證評估
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# 建立包含前處理與模型的 Pipeline
model_lr = Pipeline([("prep", preprocessor), ("regressor", LinearRegression())])
model_rf = Pipeline([("prep", preprocessor), ("regressor", RandomForestRegressor(n_estimators=100, random_state=42))])

# 執行 5 折交叉驗證 (以 R² 為評估指標)
scores_lr = cross_val_score(model_lr, X_train, y_train, cv=5, scoring="r2")
scores_rf = cross_val_score(model_rf, X_train, y_train, cv=5, scoring="r2")

print(f"[多元線性迴歸] 5折交叉驗證平均 R²: {scores_lr.mean():.4f} (標準差: {scores_lr.std():.4f})")
print(f"[隨機森林迴歸] 5折交叉驗證平均 R²: {scores_rf.mean():.4f} (標準差: {scores_rf.std():.4f})")

#### 💬【步驟 4 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將前處理管線與演算法焊接成整合式 `Pipeline`。利用「5 折交叉驗證 (5-Fold Cross Validation)」輪流以不同切分考驗模型穩定性。多元線性迴歸與隨機森林的交叉驗證 R² 皆高達約 0.998。

---
💡 **學生應該掌握的核心觀念與 AI 提問清單（依你的學習需求自由複製發問）：**

1. 📌 **K 折交叉驗證原理**
   > 「什麼是 5 折交叉驗證 (5-Fold Cross-Validation)？它是如何運作的？為什麼交叉驗證算出的平均分數比只看單一次切分的測試成績更具有客觀說服力？」

2. 📌 **多元線性迴歸的基本原理**
   > 「多元線性迴歸 (Linear Regression) 的數學預測公式長什麼樣子？模型訓練出來的權重係數 (Weights) 與截距 (Intercept) 代表什麼具體意義？」

3. 📌 **隨機森林迴歸 (Random Forest) 原理**
   > 「隨機森林是由多棵決策樹構成的集成演算法。它是如何透過 Bagging（袋裝法）與隨機特徵抽樣，達到比單一決策樹更穩定且不易過擬合的效果？」

4. 📌 **模型選型決策：複雜度 vs. 可解釋性 (Interpretability)**
   > 「當簡單的多元線性迴歸與複雜的隨機森林在 R² 分數上打成平手（皆約 0.998）時，在企業實務中通常會優先選擇哪一個？為什麼『模型可解釋性』對行銷主管做出預算分配決策如此關鍵？」

5. 🛠️ **進階探索改寫 Prompt**
   > 「請幫我加入第三個迴歸模型：『Ridge 嶺迴歸（L2 正則化）』或『Lasso（L1 正則化）』，並計算它的 5 折交叉驗證 R² 得分進行三者對比。」


---  
### 【步驟 5】：測試集正式評估與指標解讀 (MAE, RMSE, R²)
執行下方儲存格，在封存的「測試集 (X_test, y_test)」上進行最終大考，計算評估指標並繪製「真實值 vs. 預測值」散佈圖。

In [ ]:
# [步驟 5 程式碼] 測試集大考與指標計算
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# 擬合訓練集並在測試集上推論
model_lr.fit(X_train, y_train)
y_pred_lr = model_lr.predict(X_test)

# 計算評估指標
eval_results = pd.DataFrame({
    "評估指標": ["MAE (平均絕對誤差)", "RMSE (均方根誤差)", "R² (判定係數)"],
    "多元線性迴歸數值": [
        f"{mean_absolute_error(y_test, y_pred_lr):.2f} 萬元",
        f"{root_mean_squared_error(y_test, y_pred_lr):.2f} 萬元",
        f"{r2_score(y_test, y_pred_lr):.4f}"
    ]
})
display(eval_results)

# 繪製真實值 vs 預測值散佈圖
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_lr, alpha=0.5, color="royalblue", label="測試集樣本")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2, label="完美預測線 (y=x)")
plt.title("測試集預測效果評估 (真實銷售額 vs. 模型預測額)", fontsize=13)
plt.xlabel("實際銷售額 (萬元)", fontsize=11)
plt.ylabel("模型預測銷售額 (萬元)", fontsize=11)
plt.legend()
plt.tight_layout()
plt.show()

#### 💬【步驟 5 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  拿出封存未參與訓練的測試集進行最終大考，計算三項核心指標（MAE 2.38 萬元、RMSE 2.95 萬元、R² 0.999），並繪製「真實銷售額 vs. 預測銷售額」評估散佈圖。

---
💡 **學生應該掌握的核心觀念與 AI 提問清單（依你的學習需求自由複製發問）：**

1. 📌 **三大迴歸指標生活化解析 (MAE, RMSE, R²)**
   > 「請用最生活化的白話例子解釋：MAE（平均絕對誤差）、RMSE（均方根誤差）與 R²（判定係數）有何不同？如果我要向非技術背景的老闆報告模型的誤差，我應該優先挑選哪一個指標？」

2. 📌 **RMSE 對極端離群值的敏感性**
   > 「為什麼在數學上 RMSE 通常會大於或等於 MAE？當一個模型的 RMSE 遠高於 MAE 時，代表資料或預測結果中可能隱含了什麼問題？」

3. 📌 **R² 判定係數的深度剖析**
   > 「R² 最高只能是 1.0 嗎？R² 有可能是負數嗎？如果 R² 是負數代表這個模型發生了什麼事？本範例的 R² 高達 0.999 是什麼意思？」

4. 📌 **殘差分析 (Residual Analysis) 與對角線意義**
   > 「在測試集評估散佈圖中，為什麼所有數據點越貼近紅色虛線 (y=x) 代表預測越精確？什麼是『殘差 (Residuals)』？如果殘差分佈出現喇叭形狀代表什麼統計問題？」

5. 🛠️ **進階探索改寫 Prompt**
   > 「請幫我寫一段代碼，計算測試集上每一樣本的殘差 (`y_test - y_pred`)，並繪製『殘差直方圖』與『殘差散佈圖』，檢查殘差是否呈現常態分佈且均勻圍繞在 0 附近。」


---  
### 【步驟 6】：實務情境模擬預測 (Inference)
執行下方儲存格，建立一個方便重用的 `predict_sales` 函式。你可以輸入任意廣告預算組合，讓模型即時預估銷售總額！

In [ ]:
# [步驟 6 程式碼] 自訂預算情境試算函式
def predict_sales(pipeline, plan_dict):
    """輸入新廣告配置字典，自動透過 Pipeline 輸出預測銷售額"""
    input_df = pd.DataFrame([plan_dict])
    pred = pipeline.predict(input_df)[0]
    return pred

# 試算兩種不同的行銷投放提案
plan_A = {"電視廣告": 85.0, "廣播廣告": 25.0, "社群媒體廣告": 5.0, "網紅廣告": "大於百萬"}
plan_B = {"電視廣告": 40.0, "廣播廣告": 10.0, "社群媒體廣告": 2.0, "網紅廣告": "小於一萬"}

pred_A = predict_sales(model_lr, plan_A)
pred_B = predict_sales(model_lr, plan_B)

print("=== 商業廣告方案預測比較 ===")
print(f"提案 A (高預算加碼電視與大網紅) -> 預期銷售額: {pred_A:.2f} 萬元")
print(f"提案 B (低預算保守投放方案)     -> 預期銷售額: {pred_B:.2f} 萬元")
print(f"➡️ 提案 A 預計比 提案 B 多帶進 {pred_A - pred_B:.2f} 萬元的營收！")

#### 💬【步驟 6 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將訓練好的完整 Pipeline 封裝成推論函式 `predict_sales`。直接傳入原始廣告預算字典，系統自動完成前處理並輸出預期銷售額，進行提案 A（電視加碼）與提案 B（保守投放）的商業成效對比。

---
💡 **學生應該掌握的核心觀念與 AI 提問清單（依你的學習需求自由複製發問）：**

1. 📌 **推論管線 (Inference Pipeline) 的健全性**
   > 「當我們拿到完全未知的『未來全新廣告預算數據』時，Pipeline 是如何確保它自動執行與訓練時完全相同的標準化與 One-Hot 編碼轉換的？」

2. 📌 **行銷預算邊際效益評估 (ROI 決策)**
   > 「根據我們訓練出的線性迴歸模型係數，如果行銷團隊下個月想額外追加 10 千元的預算，投入在『電視廣告』跟投入在『廣播廣告』上，哪一個能帶來更高的邊際銷售額？該如何推算？」

3. 📌 **模型上線與落地部署形式**
   > 「在實際企業應用中，資料科學家通常會如何把這個訓練好的 `Pipeline` 打包成可供業務同仁使用的產品？（例如導出成 `.joblib`/`.pkl` 檔、製作成桌面視窗程式、或是串接 FastAPI 提供 RESTful API？）」

4. 🛠️ **進階探索改寫 Prompt**
   > 「請幫我寫一個 Python 互動式輸入介面（使用迴圈與 `input()`），讓業務同仁可以直接在終端機輸入電視、廣播、社群費用與網紅規模，系統自動驗證數值並即時印出預估銷售額與投資回報倍數。」
